# Evaluate bo20 retrieval recall accuracy with Docling and Milvus

In this notebook, we'll get the end-to-end recall accuracy of a retrieval pipeline made up of Docling's extraction and embedding tasks and a Milvus vector database (VDB).

Refer to the [Download Bo20 and Bo767](digital_corpora_download.ipynb) notebook to fetch the PDF documents.
In this script, you just need the small dataset **Bo20** with 20 PDFs. Since the dataset is small, the notebook uses [Milvus Lite](https://milvus.io/docs/milvus_lite.md) as vector database.

## Settings

### Environment and dependencies

Create and activate a dedicated environment:

```shell
uv venv --python 3.12 .venv && source .venv/bin/activate && uv pip install ipykernel --upgrade
```

Install the depdencies:

In [69]:
import os
import logging
logging.getLogger().setLevel(logging.WARNING)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

! uv pip install -q --upgrade docling ipython ipywidgets "pymilvus[milvus-lite]" openai

## Convertion & Ingestion

In [ ]:
# Fetch the files
from pathlib import Path

pdf_files = list(Path("../data/bo20").rglob("*.pdf"))

In [ ]:
# Convert with Docling (standard pipeline)
import time
from docling_core.types import DoclingDocument
from docling.datamodel.base_models import ConversionStatus
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
start_time = time.time()
conv_results = converter.convert_all(pdf_files, raises_on_error=False)
docs: list[DoclingDocument] = [item.document for item in conv_results if item.status == ConversionStatus.SUCCESS]
end_time = time.time() - start_time

print(f"Converted {len(docs)} out of {len(pdf_files)} files in {end_time:.2f} seconds.")

2025-11-25 11:25:37,382 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-25 11:25:37,415 - INFO - Going to convert document batch...
2025-11-25 11:25:37,415 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-25 11:25:37,420 - INFO - Loading plugin 'docling_defaults'
2025-11-25 11:25:37,424 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-25 11:25:37,427 - INFO - Loading plugin 'docling_defaults'
2025-11-25 11:25:37,440 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-25 11:25:41,806 - INFO - Auto OCR model selected ocrmac.
2025-11-25 11:25:41,815 - INFO - Accelerator device: 'mps'
2025-11-25 11:25:51,322 - INFO - Accelerator device: 'mps'
2025-11-25 11:25:51,823 - INFO - Processing document 1238975.pdf
2025-11-25 11:25:55,859 - INFO - Finished converting document 1238975.pdf in 18.48 sec.
2025-11-25 11:25:55,861 - INFO - detected format

Converted 20 out of 20 files in 265.09 seconds.


In [5]:
# Optional: save results
import os
import pickle

with open("bo20_docling_results.pkl", "wb") as filehandler:
    pickle.dump(docs, filehandler)

# Optional: save documents in JSON format
os.makedirs("docs", exist_ok=True)
for item in docs:
    item.save_as_json((Path("docs") / item.origin.filename).with_suffix(".json"))

In [6]:
# Optional: load results
import pickle

with open("bo20_docling_results.pkl", "rb") as filehandler:
    docs = pickle.load(filehandler)

In [ ]:
# Set up the embeddings with OpenAI and Nvidia API
from typing import Literal
from openai import OpenAI

client = OpenAI(
  api_key="nvapi-xxxxx",
  base_url="https://integrate.api.nvidia.com/v1"
)

def emb_text(text: str, input_type: Literal["passage", "query"]):
    return (
        client.embeddings.create(
            model="nvidia/llama-3.2-nv-embedqa-1b-v2",
            encoding_format="float",
            extra_body={"input_type": input_type, "truncate": "NONE"},
            input=text,
        ).data[0].embedding
    )

test_embedding = emb_text("This is a test", input_type="passage")
embedding_dim = len(test_embedding)
print(f"Embedding size is {embedding_dim}")

2025-11-25 11:30:18,455 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"


Embedding size is 2048


In [8]:
# Set up Docling's chunkers: page chunker and hybrid chunker using llama 3.2 tokenizer
from transformers import AutoTokenizer
from docling.chunking import HybridChunker
from docling_core.transforms.chunker import PageChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

page_chunker = PageChunker()

from_pretrained = AutoTokenizer.from_pretrained("nvidia/llama-nemotron-embed-1b-v2")
# max tokens needs to be set explicitly: 8192 tokens accoring to model card
tokenizer = HuggingFaceTokenizer(
    tokenizer=from_pretrained,
    max_tokens = 8192
)
hybrid_chunker = HybridChunker(tokenizer=tokenizer)

In [ ]:
# Chunk text
import statistics
from docling_core.transforms.chunker import DocChunk

page_chunks: list[DocChunk] = []
page_num: list[str] = []
hybrid_chunks: list[DocChunk] = []
hybrid_num: list[int] = []
for doc in docs:
    doc_chunks = list(hybrid_chunker.chunk(doc))
    hybrid_chunks.extend(doc_chunks)
    hybrid_num.append(len(doc_chunks))

    doc_chunks = list(page_chunker.chunk(doc))
    page_chunks.extend(doc_chunks)
    page_num.append(len(doc_chunks))


print(f"Hybrid chunks in {len(docs)} docs: {sum(hybrid_num)=}, {statistics.mean(hybrid_num)=}, {statistics.median(hybrid_num)=}, {max(hybrid_num)=}, {min(hybrid_num)=}")
print(f"Page chunks in {len(docs)} docs: {sum(page_num)=}, {statistics.mean(page_num)=}, {statistics.median(page_num)=}, {max(page_num)=}, {min(page_num)=}")

Hybrid chunks in 20 docs: sum(hybrid_num)=475, statistics.mean(hybrid_num)=23.75, statistics.median(hybrid_num)=17.0, max(hybrid_num)=99, min(hybrid_num)=2
Page chunks in 20 docs: sum(page_num)=436, statistics.mean(page_num)=21.8, statistics.median(page_num)=15.5, max(page_num)=116, min(page_num)=1


In [ ]:
# Set ingestion: for this simple dataset, we will use Milvus Lite
# Only dense embeddings (to be aligned with bo20_recall.ipynb)
from pymilvus import MilvusClient

#CONSISTENCY = "Bounded"
CONSISTENCY = "Strong"  # only value supported by Milvus Lite
INDEX_TYPE = "FLAT"
METRIC_TYPE = "L2"

milvus_client = MilvusClient(uri="./milvuslite_docling.db")
collection_base = "bo20_docling"
collections = {"section", "page"}
for item in collections:
    collection_name = f"{collection_base}_{item}"
    if milvus_client.has_collection(collection_name):
        milvus_client.drop_collection(collection_name)

    # taking milvus index parameters from NV Ingest client
    milvus_client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim,
        metric_type=METRIC_TYPE,
        index_type=INDEX_TYPE,
        consistency_level=CONSISTENCY,
    )

In [28]:
# Insert data
import pickle
from tqdm import tqdm

for item in collections:
    data = []
    chunks = hybrid_chunks if item == "section" else page_chunks
    for i, chunk in enumerate(tqdm(chunks, desc=f"Embedding {item} chunks")):
        embedding = emb_text(chunk.text, "passage")
        data.append({"id": i, "vector": embedding, "text": chunk.text, "meta": chunk.meta.export_json_dict()})
    milvus_client.insert(collection_name=f"{collection_base}_{item}", data=data)
    # save chunks to disk
    with open(f"{collection_base}_{item}_chunks.pkl", "wb") as f:
        pickle.dump(data, f)

Embedding section chunks:   4%|▍         | 20/475 [00:04<01:18,  5.79it/s]2025-11-25 14:12:57,689 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2025-11-25 14:12:57,690 - INFO - Retrying request to /embeddings in 0.446680 seconds
2025-11-25 14:12:58,257 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
Embedding section chunks:   5%|▌         | 24/475 [00:05<01:45,  4.28it/s]2025-11-25 14:12:58,923 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2025-11-25 14:12:58,923 - INFO - Retrying request to /embeddings in 0.382498 seconds
2025-11-25 14:12:59,433 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
Embedding section chunks:   6%|▌         | 28/475 [00:06<01:42,  4.36it/s]2025-11-25 14:13:00,073 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 4

## Evaluation

In [86]:
# Supporting function to calculate the evaluation metrics: Recal@K, only for questions on bo20 documents
import numpy as np
import pandas as pd
from collections import defaultdict

def get_recall_scores(query_df: pd.DataFrame, collection_name: str):
    # filter the queries on PDFs for this collection only
    pdf_names = [item.name for item in pdf_files]
    eval_queries = list(zip(
        query_df[query_df["pdf"].isin(pdf_names)]["query"].index,
        query_df[query_df["pdf"].isin(pdf_names)]["query"]))
    print(f"Evaluation metrics on {len(eval_queries)} questions")
    if len(eval_queries) == 0:
        return

    hits = defaultdict(list)
    query_embeddings = [emb_text(question[1], "query") for question in eval_queries]
    all_answers = milvus_client.search(
        collection_name=collection_name,
        data=query_embeddings,
        limit=10,
        consistency_level=CONSISTENCY,
        search_params={"ef": 100},
        output_fields=["meta"],
    )
    for idx, query in enumerate(eval_queries):
        expected_pdf_page = query_df["pdf_page"][query[0]]
        retrieved_answers = all_answers[idx]
        retrieved_pdfs = [result['entity']['meta']['origin']['filename'].removesuffix('.pdf') for result in retrieved_answers]
        # Warning: we assume that the first provenance of the first doc item holds the 'right' page number
        # Docling pages are 1-based
        retrieved_pages = [result['entity']['meta']['doc_items'][0]['prov'][0]['page_no'] - 1 for result in retrieved_answers]
        retrieved_pdf_pages = [f"{pdf}_{page}" for pdf, page in zip(retrieved_pdfs, retrieved_pages)]

        for k in [1, 3, 5, 10]:
            hits[k].append(expected_pdf_page in retrieved_pdf_pages[:k])
    
    for k in hits:
        print(f'  - Recall @{k}: {np.mean(hits[k]) :.3f}')

### Text Recall

In [87]:
import pandas as pd

df_query = pd.read_csv('../data/text_query_answer_gt_page.csv')
#df_query.pdf = df_query.pdf.apply(lambda x: x.replace('.pdf',''))
df_query['pdf_page'] = df_query.apply(lambda x: f"{x.pdf.removesuffix('.pdf')}_{x.gt_page}", axis=1) 
df_query

,pdf,query,answer,gt_page,pdf_page
0,1102434.pdf,How much was the ARtillery Intelligence projec...,$4.2 billion,19,1102434_19
1,1102434.pdf,How much revenue of AR advertising is expected...,$8.8 billion,3,1102434_3
2,1096078.pdf,What types of statistics were utilized by Rein...,descriptive statistics,3,1096078_3
3,1054125.pdf,What was the maximum amount requested for cond...,"$35,000.00",1,1054125_1
4,1246906.pdf,What is the median household income for the Ci...,"$53,278",7,1246906_7
...,...,...,...,...,...
483,2089825.pdf,Under the Climate Action and Low Carbon Develo...,Denis Naughten TD,0,2089825_0
484,2089825.pdf,How many organizations make up Stop Climate Ch...,30,5,2089825_5
485,2098077.pdf,What is the maximum length of Sai Yok bent-­to...,2.4 inches,1,2098077_1
486,2098077.pdf,What characteristic sets the Sai Yok Bent-toed...,enlarged thigh scales,1,2098077_1


In [88]:
for item in collections:
    collection_name = f"{collection_base}_{item}"
    print(f"Evaluation of {collection_name} on Text")
    get_recall_scores(df_query, collection_name)

Evaluation of bo20_docling_section on Text
Evaluation metrics on 3 questions
  - Recall @1: 1.000
  - Recall @3: 1.000
  - Recall @5: 1.000
  - Recall @10: 1.000
Evaluation of bo20_docling_page on Text
Evaluation metrics on 3 questions
  - Recall @1: 1.000
  - Recall @3: 1.000
  - Recall @5: 1.000
  - Recall @10: 1.000


### Table recall

In [89]:
df_query = pd.read_csv('../data/table_queries_cleaned_235.csv')[['query','pdf','page','table']]
df_query['pdf_page'] = df_query.apply(lambda x: f"{x.pdf}_{x.page}", axis=1)
df_query

,query,pdf,page,table,pdf_page
0,How much did Pendleton County spend out of the...,1003421,2,1003421_2_0,1003421_2
1,How many units are occupied by single families...,1008059,6,1008059_6_1,1008059_6
2,"In the Klamath county, what is the total valua...",1008059,6,1008059_6_1,1008059_6
3,How much did Nalco pay GRIDCO for electricity ...,1011810,21,1011810_21_0,1011810_21
4,How much coal is used at Alumina refinery of N...,1011810,21,1011810_21_2,1011810_21
...,...,...,...,...,...
230,How much is the rental income from water plant...,2407280,30,2407280_30_0,2407280_30
231,In 2020 how much were the supplemental taxes f...,2415001,65,not detected,2415001_65
232,"As of 2020, what is the total of collections a...",2415001,65,not detected,2415001_65
233,What was the net gain from the operations of t...,2416020,84,2416020_84_0,2416020_84


In [90]:
for item in collections:
    collection_name = f"{collection_base}_{item}"
    print(f"Evaluation of {collection_name} on Tables")
    get_recall_scores(df_query, collection_name)

Evaluation of bo20_docling_section on Tables
Evaluation metrics on 0 questions
Evaluation of bo20_docling_page on Tables
Evaluation metrics on 0 questions
